# AML-IDS experiments on Kaggle

**Before running:** Settings → Accelerator = *GPU T4 x2* (or P100) · Internet = *On* (for `pip install river`).
Add Data: (1) the uploaded code dataset containing `aml-ids/`, (2) CICIDS2017 with the **original per-day CSV files**,
(3) CSE-CIC-IDS2018 (per-day CSVs), (4) UNSW-NB15 (training/testing-set CSVs), (5) NSL-KDD (`KDDTrain+.txt`, `KDDTest+.txt`).
To **resume** after a 12-hour session ends, also add the *output of your previous version of this notebook*.

Run with **Save Version → Save & Run All (Commit)** so it keeps running after you close the browser.
Set `STAGE` below, one stage per run; rerun the same stage if it did not finish — finished folds are skipped.

In [ ]:
STAGE = 1          # 1: CICIDS2017 | 2: UNSW-NB15 + NSL-KDD | 3: EWC/buffer/stream | 4: transfer + ablation | 5: no-SMOTE | 6-7: UNSW shuffled
FAST = '--steps-per-epoch 1000 --max-epochs 60 --patience 10'   # report these values in Table 3
MAX_ROWS_2018 = 3000000   # CSE-CIC-IDS2018 is ~16M rows; stratified subsample to fit RAM

In [ ]:
import glob, os, shutil, subprocess, sys, time, yaml
!pip install -q river
src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/aml-ids/run.py', recursive=True)]
assert src, 'Code dataset not found: upload aml-ids-code.zip as a Kaggle dataset and add it'
W = '/kaggle/working/aml-ids'
if not os.path.exists(W):
    shutil.copytree(src[0], W, ignore=shutil.ignore_patterns('results'))
os.chdir(W)
# resume: bring back results from a previous version's output, if attached
for old in glob.glob('/kaggle/input/**/aml-ids/results', recursive=True):
    shutil.copytree(old, 'results', dirs_exist_ok=True); print('restored', old)
import torch; NGPU = torch.cuda.device_count(); print('GPUs:', NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
# locate dataset files automatically and write the config
csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
c17 = [f for f in csvs if 'workinghours' in f.lower()]
pref = [f for f in c17 if 'trafficlabelling' in f.lower() or 'generatedlabelledflows' in f.lower()]
c17 = pref or c17                     # prefer files with IP columns; never mix both versions
c18 = [f for f in csvs if '-2018' in f and 'trafficforml' in f.lower()] or [f for f in csvs if '-2018' in f]
unsw = [f for f in csvs if os.path.basename(f).lower() in ('unsw_nb15_training-set.csv', 'unsw_nb15_testing-set.csv')]
ktr = glob.glob('/kaggle/input/**/KDDTrain+.txt', recursive=True)
kte = glob.glob('/kaggle/input/**/KDDTest+.txt', recursive=True)
cfg = yaml.safe_load(open('configs/datasets.yaml'))
cfg['cicids2017']['files'] = c17; cfg['cicids2018']['files'] = c18; cfg['unsw_nb15']['files'] = unsw
if ktr: cfg['nslkdd']['train'], cfg['nslkdd']['test'] = ktr[0], kte[0]
cfg['cicids2017']['file_order'] = ['Monday', 'Tuesday', 'Wednesday', 'Thursday-WorkingHours-Morning', 'Thursday-WorkingHours-Afternoon', 'Friday-WorkingHours-Morning', 'PortScan', 'DDos']
cfg['cicids2017']['file_order'] = ['Monday', 'Tuesday', 'Wednesday', 'Thursday-WorkingHours-Morning',
    'Thursday-WorkingHours-Afternoon', 'Friday-WorkingHours-Morning', 'PortScan', 'DDos']
yaml.safe_dump(cfg, open('configs/datasets.yaml', 'w'))
print(f'CICIDS2017: {len(c17)} files (expect 8)'); print(f'CIC-IDS2018: {len(c18)} files (expect 10)')
print(f'UNSW-NB15: {len(unsw)} files (expect 2)'); print('NSL-KDD:', bool(ktr and kte))
for f in c17: print('  ', f)

In [ ]:
START = time.time()
BUDGET_H = 11.0   # stop cleanly before Kaggle's 12-hour limit so outputs are saved

def left():
    return BUDGET_H * 3600 - (time.time() - START)

def run(jobs):
    """jobs: list of argument strings; run in parallel, one per GPU (CPU jobs start with 'cpu:')."""
    procs, g = [], 0
    for j in jobs:
        if left() < 1800:
            print('time budget reached, not started:', j); continue
        env = dict(os.environ)
        if j.startswith('cpu:'):
            j = j[4:]; env['CUDA_VISIBLE_DEVICES'] = ''
        else:
            env['CUDA_VISIBLE_DEVICES'] = str(g % max(NGPU, 1)); g += 1
        name = j.split()[0] + '_' + '_'.join(w for w in j.split()[1:] if not w.startswith('-'))[:60]
        os.makedirs('logs', exist_ok=True)
        log = open(f'logs/{name}_{int(time.time())}.log', 'w')
        print('start:', j, f'({left() / 3600:.1f} h left)', flush=True)
        procs.append((j, subprocess.Popen(f'{sys.executable} run.py {j}', shell=True, env=env,
                                          stdout=log, stderr=subprocess.STDOUT)))
    for j, p in procs:
        while p.poll() is None:
            if left() < 600:
                p.terminate(); p.wait(); print('STOPPED (time budget):', j, flush=True); break
            time.sleep(30)
        else:
            print(('OK  ' if p.returncode == 0 else 'FAIL') + ':', j, flush=True)

def pairs(jobs):              # run two GPU jobs at a time (T4 x2)
    for i in range(0, len(jobs), max(NGPU, 1)):
        run(jobs[i:i + max(NGPU, 1)])

In [ ]:
if STAGE == 1:
    D = 'cicids2017'
    run([f'cpu:classical --dataset {D} --model rf',
         f'indomain --dataset {D} --model aml_ids {FAST}',
         f'indomain --dataset {D} --model transformer --no-smote {FAST}'])
    run([f'cpu:classical --dataset {D} --model svm',
         f'indomain --dataset {D} --model cnn_lstm_attn --no-smote {FAST}',
         f'indomain --dataset {D} --model mamba --no-smote {FAST}'])
    pairs([f'indomain --dataset {D} --model lstm --no-smote {FAST}',
           f'indomain --dataset {D} --model cnn --no-smote {FAST}'])
    run([f'latency --dataset {D}'])
    run([f'branches --dataset {D}'])
    run([f'plots --dataset {D}'])

In [ ]:
if STAGE == 2:
    for D in ('unsw_nb15', 'nslkdd'):
        run([f'cpu:classical --dataset {D} --model rf',
             f'indomain --dataset {D} --model aml_ids {FAST}',
             f'indomain --dataset {D} --model transformer --no-smote {FAST}'])
        run([f'cpu:classical --dataset {D} --model svm',
             f'indomain --dataset {D} --model cnn_lstm_attn --no-smote {FAST}',
             f'indomain --dataset {D} --model mamba --no-smote {FAST}'])
        pairs([f'indomain --dataset {D} --model lstm --no-smote {FAST}',
               f'indomain --dataset {D} --model cnn --no-smote {FAST}'])
        run([f'plots --dataset {D}'])

In [ ]:
if STAGE == 3:
    pairs([f'phases --dataset cicids2017 --tag lambda --folds 3 --phases "Monday,Tuesday,Wednesday|Thursday,Friday" --lams 0,100,1000,5000,10000,50000 --buffer-sizes 0,10000 {FAST}',
           f'phases --dataset cicids2017 --tag buffer --folds 3 --phases "Monday,Tuesday|Wednesday|Friday" --lams 5000 --buffer-sizes 2000,5000,10000,20000,50000 --try-balanced {FAST}'])
    run([f'stream --dataset cicids2017 --folds 3 --balanced --configs none:none,ph:naive,ph:ewconly,ph:replay,ph:ewc,adwin:ewc {FAST}'])

In [ ]:
if STAGE == 4:
    M = f'--max-rows {MAX_ROWS_2018} --balanced'
    pairs([f'cross --source cicids2017 --target cicids2018 --binary {M} {FAST}',
           f'phases --dataset cicids2017 --tag lambda_large --folds 3 --phases "Monday,Tuesday,Wednesday|Thursday,Friday" --lams 0,100000,1000000,10000000,100000000,1000000000,10000000000 --buffer-sizes 0 {FAST}',
           f'indomain --dataset cicids2017 --model abl_attention --no-smote {FAST}',
           f'indomain --dataset cicids2017 --model abl_base --no-smote {FAST}',
           f'indomain --dataset cicids2017 --model abl_multiscale --no-smote {FAST}',
           f'indomain --dataset cicids2017 --model abl_bilstm --no-smote {FAST}'])

In [ ]:
if STAGE == 5:
    pairs([f'indomain --dataset unsw_nb15 --model abl_attention --no-smote {FAST}',
           f'indomain --dataset nslkdd --model abl_attention --no-smote {FAST}'])

In [ ]:
if STAGE == 6:
    import pandas as pd
    os.makedirs('/kaggle/working/unsw_shuffled', exist_ok=True)
    files = []
    for f in cfg['unsw_nb15']['files']:
        d = pd.read_csv(f).sample(frac=1.0, random_state=0)      # same random order in stages 6 and 7
        out = '/kaggle/working/unsw_shuffled/' + os.path.basename(f)
        d.to_csv(out, index=False); files.append(out)
    cfg['unsw_shuffled'] = dict(format='unsw', files=files)
    yaml.safe_dump(cfg, open('configs/datasets.yaml', 'w'))
    run([f'cpu:classical --dataset unsw_shuffled --model rf',
         f'indomain --dataset unsw_shuffled --model abl_attention --no-smote {FAST}',
         f'indomain --dataset unsw_shuffled --model lstm --no-smote {FAST}'])

In [ ]:
if STAGE == 7:
    import pandas as pd
    os.makedirs('/kaggle/working/unsw_shuffled', exist_ok=True)
    files = []
    for f in cfg['unsw_nb15']['files']:
        d = pd.read_csv(f).sample(frac=1.0, random_state=0)      # same random order in stages 6 and 7
        out = '/kaggle/working/unsw_shuffled/' + os.path.basename(f)
        d.to_csv(out, index=False); files.append(out)
    cfg['unsw_shuffled'] = dict(format='unsw', files=files)
    yaml.safe_dump(cfg, open('configs/datasets.yaml', 'w'))
    run([f'cpu:classical --dataset unsw_shuffled --model svm',
         f'indomain --dataset unsw_shuffled --model aml_ids {FAST}',
         f'indomain --dataset unsw_shuffled --model transformer --no-smote {FAST}'])
    pairs([f'indomain --dataset unsw_shuffled --model cnn_lstm_attn --no-smote {FAST}',
           f'indomain --dataset unsw_shuffled --model mamba --no-smote {FAST}',
           f'indomain --dataset unsw_shuffled --model cnn --no-smote {FAST}'])

In [ ]:
# summary of everything computed so far (send results/tables.md for the manuscript update)
!python run.py tables > /dev/null
!head -c 4000 results/tables.md
!ls logs | tail -20
shutil.make_archive('/kaggle/working/results_backup', 'zip', 'results')